In [1]:
import os
import cv2
from tqdm import tqdm
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [2]:
def to_thermal(source_folder):
    thermal_images = []

    # Get list of images
    image_files = [f for f in os.listdir(source_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

    # Function to apply thermal effect and return image object
    def convert_to_thermal(img_path):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Load as grayscale
        img_colored = cv2.applyColorMap(img, cv2.COLORMAP_INFERNO)  # Apply thermal effect
        return img_colored

    # Convert and store in list
    for img_file in tqdm(image_files, desc="Converting to Thermal"):
        src_path = os.path.join(source_folder, img_file)
        thermal_img = convert_to_thermal(src_path)
        thermal_images.append((img_file, thermal_img))  # Store filename + image

    print("✅ All grayscale images converted to thermal format (in memory)!")
    return thermal_images

In [3]:
def glcm_feature_extractor(dataset):
    image_dataset = pd.DataFrame()
    for image in tqdm(range(dataset.shape[0]),desc="extracing GLCM features"):  #iterate through each file 
        #print(image)
        
        df = pd.DataFrame()  #Temporary data frame to capture information for each loop.
        #Reset dataframe to blank after each loop.
        
        img = dataset[image, :,:]
    ################################################################
    #START ADDING DATA TO THE DATAFRAME
  
                
         #Full image
        #GLCM = greycomatrix(img, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4])
        GLCM = graycomatrix(img, [1], [0])       
        GLCM_Energy = graycoprops(GLCM, 'energy')[0]
        df['Energy'] = GLCM_Energy
        GLCM_corr = graycoprops(GLCM, 'correlation')[0]
        df['Corr'] = GLCM_corr       
        GLCM_diss = graycoprops(GLCM, 'dissimilarity')[0]
        df['Diss_sim'] = GLCM_diss       
        GLCM_hom = graycoprops(GLCM, 'homogeneity')[0]
        df['Homogen'] = GLCM_hom       
        GLCM_contr = graycoprops(GLCM, 'contrast')[0]
        df['Contrast'] = GLCM_contr


        GLCM2 = graycomatrix(img, [3], [0])       
        GLCM_Energy2 = graycoprops(GLCM2, 'energy')[0]
        df['Energy2'] = GLCM_Energy2
        GLCM_corr2 = graycoprops(GLCM2, 'correlation')[0]
        df['Corr2'] = GLCM_corr2       
        GLCM_diss2 = graycoprops(GLCM2, 'dissimilarity')[0]
        df['Diss_sim2'] = GLCM_diss2       
        GLCM_hom2 = graycoprops(GLCM2, 'homogeneity')[0]
        df['Homogen2'] = GLCM_hom2       
        GLCM_contr2 = graycoprops(GLCM2, 'contrast')[0]
        df['Contrast2'] = GLCM_contr2

        GLCM3 = graycomatrix(img, [5], [0])       
        GLCM_Energy3 = graycoprops(GLCM3, 'energy')[0]
        df['Energy3'] = GLCM_Energy3
        GLCM_corr3 = graycoprops(GLCM3, 'correlation')[0]
        df['Corr3'] = GLCM_corr3       
        GLCM_diss3 = graycoprops(GLCM3, 'dissimilarity')[0]
        df['Diss_sim3'] = GLCM_diss3       
        GLCM_hom3 = graycoprops(GLCM3, 'homogeneity')[0]
        df['Homogen3'] = GLCM_hom3       
        GLCM_contr3 = graycoprops(GLCM3, 'contrast')[0]
        df['Contrast3'] = GLCM_contr3

        GLCM4 = graycomatrix(img, [0], [np.pi/4])       
        GLCM_Energy4 = graycoprops(GLCM4, 'energy')[0]
        df['Energy4'] = GLCM_Energy4
        GLCM_corr4 = graycoprops(GLCM4, 'correlation')[0]
        df['Corr4'] = GLCM_corr4       
        GLCM_diss4 = graycoprops(GLCM4, 'dissimilarity')[0]
        df['Diss_sim4'] = GLCM_diss4       
        GLCM_hom4 = graycoprops(GLCM4, 'homogeneity')[0]
        df['Homogen4'] = GLCM_hom4       
        GLCM_contr4 = graycoprops(GLCM4, 'contrast')[0]
        df['Contrast4'] = GLCM_contr4
        
        GLCM5 = graycomatrix(img, [0], [np.pi/2])       
        GLCM_Energy5 = graycoprops(GLCM5, 'energy')[0]
        df['Energy5'] = GLCM_Energy5
        GLCM_corr5 = graycoprops(GLCM5, 'correlation')[0]
        df['Corr5'] = GLCM_corr5       
        GLCM_diss5 = graycoprops(GLCM5, 'dissimilarity')[0]
        df['Diss_sim5'] = GLCM_diss5       
        GLCM_hom5 = graycoprops(GLCM5, 'homogeneity')[0]
        df['Homogen5'] = GLCM_hom5       
        GLCM_contr5 = graycoprops(GLCM5, 'contrast')[0]
        df['Contrast5'] = GLCM_contr5
        
        #Add more filters as needed
        #entropy = shannon_entropy(img)
        #df['Entropy'] = entropy

        
        #Append features from current image to the dataset
        image_dataset = pd.concat([image_dataset, df], ignore_index=True)
        
    return image_dataset

In [4]:


# Load YOLOv8 model
def yolo_feature_extractor(source):
    model_detect = YOLO("best_neu.pt")
    
    # Choose the layer to hook
    features = []
    def hook(module, input, output):
        features.append(output)
    
    layer_to_hook = model_detect.model.model[-2]  # You can change this index as needed
    hook_handle = layer_to_hook.register_forward_hook(hook)
    
    # Folder path
    folder_path = source
    
    data = {}
    
    # Loop through all images
    for filename in tqdm(os.listdir(folder_path),desc="extracting yolo features"):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(folder_path, filename)
            
            # Preprocess image
            im = cv2.imread(image_path)
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
            im = cv2.resize(im, (640, 640))
            im = im.astype(np.float32) / 255.0
            im = torch.from_numpy(im.transpose(2, 0, 1)).unsqueeze(0)
            
            features.clear()  # Clear previous feature
            with torch.no_grad():
                _ = model_detect.model(im)
            
            # Extract and process features
            feat_tensor = features[0]
            feat_vector = torch.nn.functional.adaptive_avg_pool2d(feat_tensor, 1).view(feat_tensor.shape[0], -1)
            
            # Save in dictionary
            data[filename] = feat_vector.squeeze().numpy()
    
    # Convert to DataFrame
    df = pd.DataFrame.from_dict(data, orient='index')
    
    # Optional: Save to CSV
    # df.to_csv("yolo_features.csv")
    
    print(df.shape)
    print(df.head())
    
    # Remove the hook
    hook_handle.remove()
    return df


In [5]:
import cv2
from ultralytics import YOLO
model = YOLO("best_neu.pt")
def yolo_bounding_box(img_path):
    
    
    # Run prediction
    results = model(img_path, conf=0.1,verbose=False)

    # Load original image
    image = cv2.imread(img_path)
    segments = []

    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)  # (x1, y1, x2, y2)
        
        #for i, (x1, y1, x2, y2) in enumerate(boxes):
            #segment = image[y1:y2, x1:x2]  # Crop using box coordinates
            #print(f"Segment {i}: shape = {segment.shape}")
            #segments.append(segment)
    return boxes
            # Optional: save the segment
            # cv2.imwrite(f"segment_{i}.jpg", segment)


##############################################
best_neu.pt


In [ ]:
def segmentor(img,boxes):
    for i, (x1, y1, x2, y2) in enumerate(boxes):
            segment = image[y1:y2, x1:x2]  # Crop using box coordinates
            print(f"Segment {i}: shape = {segment.shape}")
            segments.append(segment)
    


In [6]:
SOURCE_FOLDER = Path("images/Testing_data/")
SOURCE_ANNOT = "labels/Testing_data/"
rois = []
for file in tqdm(SOURCE_FOLDER.glob("*"),desc="working"):
    if file.suffix.lower() in [".jpg", ".png", ".jpeg"]:
        rois.append(yolo_bounding_box(file))
thermal = to_thermal(SOURCE_FOLDER)
thermal_rois = 


#print(rois)
        


working: 180it [00:05, 33.32it/s]
Converting to Thermal: 100%|███████████████████████████████████████████████████████| 180/180 [00:00<00:00, 1668.71it/s]

✅ All grayscale images converted to thermal format (in memory)!
